# デモ5：言葉で切り抜く（OWLv2 ＋ SAM 2）

2つのモデルをつなげて、言葉で指定した物を、輪郭に沿って切り抜きます。

1. **OWLv2**（デモ2）が、言葉で指定した物を枠で探す。
2. **SAM 2**（Segment Anything Model 2）が、枠の中の物を画素単位で切り抜く。SAM は、枠や点で「どれ」を指定すると、その物の領域（マスク）を返すモデルです。何の物かは知らなくても切り抜けます。

研究では、このように**得意なことが違うモデルを組み合わせて**、1つのモデルではできない処理を作ることがよくあります（Meta の最新の SAM3 は、言葉の指定まで1つのモデルでできますが、重みの利用に申請が必要です）。

- モデル：[google/owlv2-base-patch16-ensemble](https://huggingface.co/google/owlv2-base-patch16-ensemble)（Apache-2.0）、[facebook/sam2.1-hiera-small](https://huggingface.co/facebook/sam2.1-hiera-small)（Apache-2.0、約3900万パラメータ）
- 中の Transformer：OWLv2 は ViT。SAM 2 は、画像を読む部分（Hiera）と、指定（枠や点）と画像の特徴をアテンションで突き合わせてマスクを出す部分（マスクデコーダ）からなる

**最初に**：「ファイル」→「ドライブにコピーを保存」で自分のコピーを作り、「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行します。コードを読む必要はありません。何ができて、何が苦手かを見つけるのが目的です。

## 1. 準備

目安：約3分（うち計算 約1分）

In [ ]:
import importlib.util
import subprocess
import sys

# Colab に入っていない、または古いライブラリを入れる
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage import data

device = "cuda" if torch.cuda.is_available() else "cpu"
# T4 は bfloat16 に対応していないので、GPU では float16 を使う
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

In [ ]:
from transformers import Sam2Model, Sam2Processor, pipeline

detector = pipeline("zero-shot-object-detection", model="google/owlv2-base-patch16-ensemble",
                    device=device, dtype=dtype)
sam_processor = Sam2Processor.from_pretrained("facebook/sam2.1-hiera-small")
sam = Sam2Model.from_pretrained("facebook/sam2.1-hiera-small", dtype=dtype).to(device).eval()

COLORS = np.array([[230, 60, 60], [60, 90, 230], [40, 170, 80], [230, 160, 30], [160, 60, 200]])


def segment(image, boxes=None, points=None):
    """枠（[x0, y0, x1, y1] のリスト）か点（[x, y] のリスト）で指定した物のマスクを返す（物の数×縦×横）"""
    if boxes is not None:
        inputs = sam_processor(images=image, input_boxes=[boxes], return_tensors="pt").to(device)
    else:  # 点は1つずつ別の物として扱う
        inputs = sam_processor(images=image, input_points=[[[p] for p in points]],
                               input_labels=[[[1] for _ in points]], return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype)
    with torch.no_grad():
        outputs = sam(**inputs, multimask_output=False)
    masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0]
    return masks[:, 0].numpy()


def show_masks(image, masks, labels, title=""):
    """マスクを色で重ねて表示する"""
    overlay = np.array(image).astype(float)
    for i, mask in enumerate(masks):
        overlay[mask] = overlay[mask] * 0.45 + COLORS[i % len(COLORS)] * 0.55
    plt.figure(figsize=(7, 7))
    plt.imshow(overlay.astype(np.uint8))
    plt.title(title + "  " + ", ".join(labels))
    plt.axis("off")
    plt.show()

## 2. 例の画像で試す

目安：約5分

In [ ]:
USE_UPLOAD = False  # True にすると、自分の画像をアップロードできる（JPEG か PNG）

# 例の画像（scikit-image に入っている写真。CC0 またはパブリックドメイン）
images = {
    "astronaut": Image.fromarray(data.astronaut()),
    "cat": Image.fromarray(data.chelsea()),
    "coffee": Image.fromarray(data.coffee()),
}
if USE_UPLOAD:
    import io
    from google.colab import files
    for name, content in files.upload().items():
        images[name] = Image.open(io.BytesIO(content)).convert("RGB")

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), squeeze=False)
for ax, (name, image) in zip(axes[0], images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
QUERIES = ["a cup", "a spoon", "a face", "a flag", "a cat"]  # ← 変えてみる
THRESHOLD = 0.2  # ← OWLv2 のしきい値

for name, image in images.items():
    found = detector(image, candidate_labels=QUERIES, threshold=THRESHOLD)
    if not found:
        print(name, ": 見つからなかった")
        continue
    boxes = [[r["box"]["xmin"], r["box"]["ymin"], r["box"]["xmax"], r["box"]["ymax"]] for r in found]
    masks = segment(image, boxes=boxes)
    show_masks(image, masks, [r["label"] for r in found], name)

### 点で指定する

SAM 2 は、枠の代わりに点（画像の左上を原点とした x, y 座標）でも指定できます。点を置いた所の物が切り抜かれます。2. の図の目盛りを見て、座標を変えてみましょう。

In [ ]:
POINTS = [[180, 120], [300, 400]]  # ← 変えてみる（[x, y] のリスト。点ごとに別の物）

image = images["astronaut"]
masks = segment(image, points=POINTS)
show_masks(image, masks, [f"point {p}" for p in POINTS], "astronaut")

## 3. やってみよう

目安：約10分

- 自分の写真で、研究で切り抜きたい物（部品、細胞、道路の傷など）を言葉で指定して試します。OWLv2 が見つけられないときは、点で指定するとどうでしょうか。
- 切り抜いた結果を使って、背景だけ白くする、物の面積（マスクの画素数）を数える、などの処理を考えてみましょう。マスクは True と False の配列なので、`masks[0].sum()` で画素数がわかります。